# HFSS fixed-input debug run

This temporary debug notebook uses the supplied Monte Carlo parameter inputs, in their original order, for HFSS calls. It keeps the central-difference notebook's 79-call control flow only as an HFSS debug harness: the first 79 supplied rows are used and surplus rows are discarded.

Because these fixed Monte Carlo rows are **not** symmetric central-difference points, gradient post-processing is intentionally disabled. Run the cells from top to bottom.


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
import lib_gp

%load_ext autoreload
%autoreload 2


In [ ]:
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)
backbone = lib_backbone.Backbone(config=app_config)
backbone.initStorer()
base_dir = app_config.env.dir_base
cfg = app_config.hfss

PARAM_NAMES = list(cfg.param_names)
ROUND_DECIMALS = app_config.runtime.round_decimals

# ===== Fixed debug-input settings =====
# This center, covariance, seed, and clipping rule reproduce the supplied rows.
x_center = np.asarray([
    6.225785, 4.065235, 9.188137, 9.788713, 1.0,
    1.071733, 2.0, 1.456739, 1.835538, 0.537034,
    2.0, 3.71126, 6.0,
], dtype=float)
sigma_vec = np.asarray([
    0.1, 0.1, 0.1, 0.1, 0.1,
    0.01, 0.02, 0.02, 0.02, 0.02,
    0.1, 0.1, 0.1,
], dtype=float)
DEBUG_RANDOM_SEED = 101
DEBUG_PERTURBATION_ROWS = 100
PARAM_UNITS = {name: "mm" for name in PARAM_NAMES}
perturbation_enabled = {name: True for name in PARAM_NAMES}
STEP_SCALES = [0.25, 0.5, 1.0]
REFERENCE_STEP_SCALE = 0.5
custom_step = None
repeats = 1
objective_col = "S11"
OBJECTIVE_UNIT = "dB"
OUTPUT_DIR = backbone._get_dir_run()

print(f"d={len(PARAM_NAMES)}, active={sum(perturbation_enabled.values())}")


In [ ]:
def _enabled_array(param_names, enabled):
    if isinstance(enabled, dict):
        if set(enabled) != set(param_names):
            missing = set(param_names) - set(enabled)
            extra = set(enabled) - set(param_names)
            raise ValueError(f"perturbation_enabled keys must match PARAM_NAMES; missing={missing}, extra={extra}")
        return np.asarray([bool(enabled[name]) for name in param_names])

    output = np.asarray(enabled, dtype=bool).reshape(-1)
    if output.size != len(param_names):
        raise ValueError("perturbation_enabled length must equal len(PARAM_NAMES).")
    return output


def _step_array(param_names, sigma, step_scale, custom):
    if custom is None:
        return float(step_scale) * sigma
    if isinstance(custom, dict):
        if set(custom) != set(param_names):
            raise ValueError("custom_step keys must match PARAM_NAMES exactly.")
        return np.asarray([custom[name] for name in param_names], dtype=float)

    output = np.asarray(custom, dtype=float).reshape(-1)
    if output.size != len(param_names):
        raise ValueError("custom_step length must equal len(PARAM_NAMES).")
    return output


def validate_central_difference_inputs(param_names, center, sigma, enabled, steps, round_decimals):
    center = np.asarray(center, dtype=float).reshape(-1)
    sigma = np.asarray(sigma, dtype=float).reshape(-1)
    steps = np.asarray(steps, dtype=float).reshape(-1)
    dimensions = center.size

    if len(param_names) != dimensions:
        raise ValueError("len(PARAM_NAMES) must equal len(x_center).")
    if sigma.size != dimensions:
        raise ValueError("len(sigma_vec) must equal len(x_center).")
    active = _enabled_array(param_names, enabled)
    if steps.size != dimensions:
        raise ValueError("step_vec length must equal len(x_center).")
    if np.any(sigma[active] <= 0):
        raise ValueError("Every active parameter sigma must be positive.")
    if np.any(steps[active] <= 0):
        raise ValueError("Every active parameter step must be positive.")
    if not np.all(np.isfinite(center)) or not np.all(np.isfinite(sigma)) or not np.all(np.isfinite(steps)):
        raise ValueError("center, sigma, and steps must be finite.")
    if int(round_decimals) != round_decimals or round_decimals < 0:
        raise ValueError("round_decimals must be a non-negative integer.")

    rounded_center = np.round(center, int(round_decimals))
    for parameter_index in np.flatnonzero(active):
        minus = rounded_center.copy()
        plus = rounded_center.copy()
        minus[parameter_index] -= steps[parameter_index]
        plus[parameter_index] += steps[parameter_index]
        minus = np.round(minus, int(round_decimals))
        plus = np.round(plus, int(round_decimals))
        minus_step = rounded_center[parameter_index] - minus[parameter_index]
        plus_step = plus[parameter_index] - rounded_center[parameter_index]
        if minus_step <= 0 or plus_step <= 0 or not np.isclose(minus_step, plus_step, rtol=0.0, atol=10.0 ** (-int(round_decimals)) / 2.0):
            name = param_names[parameter_index]
            raise ValueError(
                f"{name}: the configured step cannot form distinct symmetric perturbations "
                f"at {int(round_decimals)} decimal places; increase sigma/custom_step or precision."
            )

    return center, sigma, active, steps


In [ ]:
def build_central_difference_design(param_names, center, sigma, enabled,
                                    step_scale=0.5, custom_step=None, repeats=1,
                                    round_decimals=6, include_center=False):
    """Build an arbitrary-dimensional, central-difference-only design."""
    if int(repeats) != repeats or repeats < 1:
        raise ValueError("repeats must be a positive integer.")

    sigma = np.asarray(sigma, dtype=float).reshape(-1)
    requested = _step_array(param_names, sigma, step_scale, custom_step)
    center, sigma, active, requested = validate_central_difference_inputs(
        param_names, center, sigma, enabled, requested, round_decimals
    )
    rounded_center = np.round(center, round_decimals)
    logical = []

    if include_center:
        logical.append(dict(
            parameter_index=-1,
            parameter_name="__center__",
            side="center",
            requested_step=0.0,
            actual_step=0.0,
            is_center=True,
            step_scale=float(step_scale),
            _x=rounded_center,
        ))

    for parameter_index, parameter_name in enumerate(param_names):
        if not active[parameter_index]:
            continue

        minus = rounded_center.copy()
        plus = rounded_center.copy()
        minus[parameter_index] -= requested[parameter_index]
        plus[parameter_index] += requested[parameter_index]
        minus = np.round(minus, round_decimals)
        plus = np.round(plus, round_decimals)
        actual_step = plus[parameter_index] - rounded_center[parameter_index]

        for side, point in (("minus", minus), ("plus", plus)):
            logical.append(dict(
                parameter_index=parameter_index,
                parameter_name=parameter_name,
                side=side,
                requested_step=requested[parameter_index],
                actual_step=actual_step,
                is_center=False,
                step_scale=float(step_scale),
                _x=point,
            ))

    physical_points = [tuple(row["_x"]) for row in logical]
    if len(physical_points) != len(set(physical_points)):
        raise ValueError("Duplicate physical points were generated before repeats.")

    rows = []
    for item in logical:
        point = item.pop("_x")
        for _ in range(int(repeats)):
            row = dict(item)
            row.update({name: float(point[index]) for index, name in enumerate(param_names)})
            rows.append(row)
    return pd.DataFrame(rows)


def combine_step_designs(designs, share_center=True):
    """Combine scale designs and retain one physical center per repeat."""
    if not designs:
        raise ValueError("At least one step-scale design is required.")

    combined = pd.concat(designs, ignore_index=True)
    center = combined[combined["is_center"]].copy()
    noncenter = combined[~combined["is_center"]].copy()
    if share_center and len(center):
        metadata_columns = {
            "parameter_index", "parameter_name", "side", "requested_step",
            "actual_step", "is_center", "step_scale",
        }
        parameter_columns = [column for column in combined.columns if column not in metadata_columns]
        if len(center[parameter_columns].drop_duplicates()) != 1:
            raise ValueError("Step-scale designs do not share the same physical center.")
        scale_count = center["step_scale"].nunique()
        if scale_count < 1 or len(center) % scale_count:
            raise ValueError("Every step scale must contain the same number of center repeats.")
        repeat_count = len(center) // scale_count
        center = center.iloc[:repeat_count].copy()
        center["step_scale"] = np.nan

    output = pd.concat([center, noncenter], ignore_index=True)
    return output.sort_values(
        ["is_center", "parameter_index", "step_scale", "side"],
        ascending=[False, True, True, True],
        na_position="first",
    ).reset_index(drop=True)


## Fixed HFSS debug inputs

The central-difference design is built only to preserve the original notebook's call count and metadata layout. Before saving or running it, its physical parameter columns are replaced, in order, with the supplied Monte Carlo sequence. The sequence contains more rows than the 79-call design, so only its leading rows are used.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
designs = []
for scale in STEP_SCALES:
    designs.append(build_central_difference_design(
        PARAM_NAMES,
        x_center,
        sigma_vec,
        perturbation_enabled,
        scale,
        custom_step,
        repeats,
        ROUND_DECIMALS,
        include_center=True,
    ))

design_df = combine_step_designs(designs, share_center=True)

bounds = list(zip(np.asarray(cfg.lower_bounds, dtype=float), np.asarray(cfg.upper_bounds, dtype=float)))
rng = np.random.default_rng(DEBUG_RANDOM_SEED)
debug_perturbations = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=np.diag(sigma_vec ** 2),
    n_samples=DEBUG_PERTURBATION_ROWS,
    bounds=bounds,
    rng=rng,
)
debug_parameter_inputs = np.vstack([x_center, debug_perturbations])
debug_parameter_inputs = np.round(debug_parameter_inputs, ROUND_DECIMALS)

if len(debug_parameter_inputs) < len(design_df):
    raise ValueError(
        f"Only {len(debug_parameter_inputs)} fixed debug rows are available for "
        f"{len(design_df)} HFSS evaluations."
    )

# Use supplied inputs from the top; intentionally discard any surplus rows.
design_df.loc[:, PARAM_NAMES] = debug_parameter_inputs[:len(design_df)]
design_df.to_csv(OUTPUT_DIR / "central_difference_input.csv", index=False)
print("Predicted HFSS evaluations:", len(design_df))
print("Available fixed debug rows:", len(debug_parameter_inputs))
print("Discarded surplus rows:", len(debug_parameter_inputs) - len(design_df))
display(design_df)


In [ ]:
def getResult(temp_hfss_path):
    try:
        df_temp = pd.read_csv(temp_hfss_path)
        s11_value = df_temp.iloc[-1, -1]

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return True, float(s11_value)

    except Exception as error:
        print(f"[Error][getResult] Failed to process result: {error}")
        return False, np.nan


def evaluate_central_difference_design(design, param_names, backbone, config_raw, app_config,
                                       output_dir, objective_col="S11"):
    """Evaluate the complete design once; checkpoint and resume behavior is intentionally absent."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _, model_paths_str = backbone._get_path_models()
    temp_file = str(output_dir / Path(config_raw["io"]["filename_temp"]))
    run_config = {
        "n_simulation": int(len(design)),
        "n_repeats": 1,
        "WATCH_DIR": str(output_dir),
        "INPUT_FILE": str(output_dir / Path(config_raw["io"]["filename_input"])),
        "MODEL_FILE": model_paths_str,
        "RESULTS_FILE": str(output_dir / Path(config_raw["io"]["filename_output"])),
        "TEMP_FILE": temp_file,
        "DONE_FLAG_FILE": str(output_dir / "hfss.done"),
    }
    done_flag_path = Path(run_config["DONE_FLAG_FILE"])
    done_flag_path.unlink(missing_ok=True)

    config_json = app_config.env.dir_base / "_config_HFSS.json"
    with open(config_json, "w") as file:
        json.dump(run_config, file, indent=4)

    records = []
    try:
        for _, metadata in design.iterrows():
            params = metadata[param_names].to_numpy(dtype=float)
            sim_id = backbone._getSimulationID()
            backbone.call_subroutine(
                run_config,
                sim_id,
                param_names,
                params,
                value_fmt=f"{{:.{app_config.runtime.round_decimals}f}}",
            )
            success, s11_value = getResult(temp_file)
            if not success:
                raise RuntimeError(f"HFSS result processing failed for parameters {params.tolist()}.")
            record = metadata.to_dict()
            record[objective_col] = s11_value
            records.append(record)
    finally:
        done_flag_path.touch()
        config_json.unlink(missing_ok=True)

    results = pd.DataFrame(records)
    results.to_csv(output_dir / "central_difference_results.csv", index=False)
    return results


results_df = evaluate_central_difference_design(
    design_df, PARAM_NAMES, backbone, _config, app_config, OUTPUT_DIR, objective_col
)


## Gradient post-processing disabled

The fixed debug rows are Monte Carlo inputs rather than symmetric axis perturbations. The metadata columns exist only to retain the original 79-call harness, so interpreting the results as central differences would be incorrect.


In [ ]:
print("Debug mode: central-difference gradient calculation is intentionally disabled.")


In [ ]:
results_path = OUTPUT_DIR / "central_difference_results.csv"
if results_path.exists():
    print(f"Saved fixed-input HFSS debug results: {results_path}")
    print("No gradient outputs were generated because the supplied rows are not central-difference pairs.")
